In [1]:
import numpy as np
import pandas as pd
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder

In [2]:
mnist = fetch_openml('mnist_784', version=1)

In [3]:
df = pd.concat([mnist.data,mnist.target],axis=1)

In [4]:
df.head()

,pixel1,pixel2,pixel3,pixel4,pixel5,pixel6,pixel7,pixel8,pixel9,pixel10,...,pixel776,pixel777,pixel778,pixel779,pixel780,pixel781,pixel782,pixel783,pixel784,class
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,5
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,4
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,9


In [5]:
df.shape

(70000, 785)

# **MY OWN CODE BELOW**

In [6]:
class neural_network:

    def __init__(self,layers: list[int],nodes: list[int],activations: list[str],features: int):

        layers = np.asarray(layers)
        nodes = np.asarray(nodes)
        activations = np.asarray(activations)

        if len(activations) <= 0 or len(layers) <= 0 or len(nodes) <= 0:
            raise ValueError("The input is incomplete")
        
        elif activations[-1] not in ['softmax','linear','sigmoid']:
            raise ValueError("The last node must have a valid activation")

        elif len(layers) != len(nodes):
            raise ValueError("Please input valid size layer and valid number of nodes")

        elif len(nodes) != len(activations):
            raise ValueError("Number of nodes and activations must match")

        elif layers.ndim != 1 or nodes.ndim != 1 or activations.ndim != 1:
            raise ValueError("layers, nodes, and activations must be 1D vectors")

        self.layers = layers
        self.nodes = nodes
        self.activations = activations
        self.features = features
        self.weights = None
        self.biases = None
        self.batch_size = 64


    # WEIGHTS INITIALIZATION
    def init_parameters(self):

        weights = []
        biases = []

        for i in range(len(self.layers)):

            if i == 0:
                fan_in = self.features
            else:
                fan_in = self.nodes[i-1]

            if self.activations[i] in ['relu','leaky_relu']:
                weights.append(np.random.randn(fan_in,self.nodes[i]) * np.sqrt(2/fan_in))
            else:
                weights.append(np.random.randn(fan_in,self.nodes[i]) * np.sqrt(1/fan_in))

            biases.append(np.zeros((self.nodes[i],1)))

        return weights,biases


    # ACTIVATIONS
    def relu(self,Z):
        return np.maximum(0,Z)

    def leaky_relu(self,Z):
        return np.maximum(0.001*Z,Z)

    def sigmoid(self,Z):
        Z = np.clip(Z,-500,500)
        return 1/(1+np.exp(-Z))

    def tanh(self,Z):
        return np.tanh(Z)

    def softmax(self,Z):
        expZ = np.exp(Z-np.max(Z,axis=1,keepdims=True))
        return expZ/np.sum(expZ,axis=1,keepdims=True)


    # LOSS FUNCTIONS
    def loss_cal(self,Y_pred,Y):

        epsilon = 1e-12

        if self.activations[-1] == 'sigmoid':
            Y_pred = np.clip(Y_pred,epsilon,1-epsilon)
            return -np.mean(Y*np.log(Y_pred)+(1-Y)*np.log(1-Y_pred))

        elif self.activations[-1] == 'softmax':
            Y_pred = np.clip(Y_pred,epsilon,1-epsilon)
            return -np.mean(np.sum(Y*np.log(Y_pred),axis=1))

        elif self.activations[-1] == 'linear':
            return np.mean((Y_pred-Y)**2)/2


    # ACCURACY
    def accuracy(self,Y_pred,Y):

        if self.activations[-1] == 'sigmoid':
            predicted_class = np.where(Y_pred >= 0.5,1,0)
            return np.mean(predicted_class == Y)

        elif self.activations[-1] == 'softmax':
            predicted_class = np.argmax(Y_pred,axis=1)
            true_class = np.argmax(Y,axis=1)
            return np.mean(predicted_class == true_class)

        elif self.activations[-1] == 'linear':
            ss_res = np.sum((Y-Y_pred)**2)
            ss_tot = np.sum((Y-np.mean(Y))**2)

            if ss_tot == 0:
                return 0

            return 1-(ss_res/ss_tot)


    # FORWARD PROPAGATION
    def forward_propagation(self,weights,biases,X):

        forward_pass = []

        for i in range(len(weights)):

            if i == 0:
                Z = (X@weights[i])+biases[i].T
            else:
                Z = (forward_pass[i-1]@weights[i])+biases[i].T

            if self.activations[i] == 'relu':
                forward_pass.append(self.relu(Z))

            elif self.activations[i] == 'leaky_relu':
                forward_pass.append(self.leaky_relu(Z))

            elif self.activations[i] == 'sigmoid':
                forward_pass.append(self.sigmoid(Z))

            elif self.activations[i] == 'tanh':
                forward_pass.append(self.tanh(Z))

            elif self.activations[i] == 'softmax':
                forward_pass.append(self.softmax(Z))

            elif self.activations[i] == 'linear':
                forward_pass.append(Z)

        return forward_pass


    # BACKWARD PROPAGATION
    def backward_propagation(self,weights,forward_pass,X,Y):

        gradients_w = []
        gradients_b = []

        batch_size = X.shape[0]

        dZ = forward_pass[-1]-Y

        for i in range(len(weights)-1,-1,-1):

            if i == 0:
                A_prev = X
            else:
                A_prev = forward_pass[i-1]

            dW = (A_prev.T@dZ)/batch_size
            db = np.sum(dZ,axis=0).reshape(-1,1)/batch_size

            gradients_w.append(dW)
            gradients_b.append(db)

            if i > 0:

                dA = dZ@weights[i].T

                if self.activations[i-1] == 'relu':
                    dZ = dA*(forward_pass[i-1] > 0)

                elif self.activations[i-1] == 'leaky_relu':
                    dZ = dA*np.where(forward_pass[i-1] > 0,1,0.001)

                elif self.activations[i-1] == 'sigmoid':
                    A = forward_pass[i-1]
                    dZ = dA*A*(1-A)

                elif self.activations[i-1] == 'tanh':
                    A = forward_pass[i-1]
                    dZ = dA*(1-A**2)

                elif self.activations[i-1] == 'linear':
                    dZ = dA

        gradients_w.reverse()
        gradients_b.reverse()

        return gradients_w,gradients_b


    # UPDATING WEIGHTS
    def update_weights(self,weights,biases,gradients_w,gradients_b,alpha=0.01):

        for i in range(len(weights)):
            weights[i] -= alpha*gradients_w[i]
            biases[i] -= alpha*gradients_b[i]

        return weights,biases


    # TRAINING
    def train_network(self,X,Y,epochs=100,alpha=0.01):

        if hasattr(X,'toarray'):
            X = X.toarray()
        else:
            X = np.asarray(X)

        if hasattr(Y,'toarray'):
            Y = Y.toarray()
        else:
            Y = np.asarray(Y)

        X = np.asarray(X,dtype=np.float64)
        Y = np.asarray(Y,dtype=np.float64)

        if X.max() > 1:
            X = X/255.0

        if X.ndim != 2:
            raise ValueError(f"X must be 2D. Got {X.shape}")

        if Y.ndim != 2:
            raise ValueError(f"Y must be 2D. Got {Y.shape}")

        if X.shape[1] != self.features:
            raise ValueError(f"X has {X.shape[1]} features, but network expects {self.features}")

        if X.shape[0] != Y.shape[0]:
            raise ValueError("X and Y must have the same number of samples")

        if Y.shape[1] != self.nodes[-1]:
            raise ValueError(f"Y has {Y.shape[1]} outputs, but network has {self.nodes[-1]} output neurons")

        weights,biases = self.init_parameters()

        samples = X.shape[0]

        for epoch in range(epochs):

            indices = np.random.permutation(samples)

            X_shuffled = X[indices]
            Y_shuffled = Y[indices]

            for start in range(0,samples,self.batch_size):

                end = min(start + self.batch_size,samples)

                X_batch = X_shuffled[start:end]
                Y_batch = Y_shuffled[start:end]

                forward_pass = self.forward_propagation(weights,biases,X_batch)

                gradients_w,gradients_b = self.backward_propagation(weights, forward_pass, X_batch, Y_batch)

                weights,biases = self.update_weights(weights, biases, gradients_w, gradients_b, alpha)

            forward_pass = self.forward_propagation(weights,biases,X)

            Y_pred = forward_pass[-1]

            loss = self.loss_cal(Y_pred,Y)
            accuracy = self.accuracy(Y_pred,Y)

            if epoch % 10 == 0:
                print("Epoch:", epoch, "Loss:", loss, "Accuracy:", accuracy)

        self.weights = weights
        self.biases = biases


    # PREDICTION
    def prediction(self,X_test):

        if self.weights is None:
            raise ValueError("The network has not been trained yet")

        if hasattr(X_test,'toarray'):
            X_test = X_test.toarray()
        else:
            X_test = np.asarray(X_test)

        X_test = np.asarray(X_test,dtype=np.float64)

        if X_test.max() > 1:
            X_test = X_test/255.0

        forward_pass = self.forward_propagation(self.weights,self.biases,X_test)

        output = forward_pass[-1]

        if self.activations[-1] == 'sigmoid':
            return np.where(output >= 0.5,1,0)

        elif self.activations[-1] == 'softmax':
            return np.argmax(output,axis=1)

        elif self.activations[-1] == 'linear':
            return output

        else:
            raise ValueError("Unsupported output activation")

## **Testing Below**

In [7]:
df.head()

,pixel1,pixel2,pixel3,pixel4,pixel5,pixel6,pixel7,pixel8,pixel9,pixel10,...,pixel776,pixel777,pixel778,pixel779,pixel780,pixel781,pixel782,pixel783,pixel784,class
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,5
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,4
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,9


In [8]:
X = df.iloc[:,:-1].to_numpy().astype(int)
y = df.iloc[:,-1].to_numpy().astype(int).reshape(-1,1)

In [9]:
X = np.asarray(X,dtype=np.float64)
y = np.asarray(y,dtype=np.float64)

In [10]:
ohe = OneHotEncoder()
y = ohe.fit_transform(y)

In [11]:
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=2)

In [44]:
model = neural_network(
    layers=[1, 2, 3],
    nodes=[10, 16, 10],
    activations=['relu', 'relu', 'softmax'],
    features=784,
)


# ==========================================
# TRAIN
# ==========================================

model.train_network(
    X_train,
    y_train,
    epochs=100,
    alpha=0.01
)


# ==========================================
# PREDICT
# ==========================================

predictions = model.prediction(X_test).reshape(14000,1)

true_classes = np.argmax(y_test, axis=1)

accuracy = np.mean(predictions == true_classes)

print("\nFinal Accuracy:", accuracy)

print("\nFirst 20 predictions:")
print(predictions.reshape(14000)[:20])

print("\nFirst 20 actual:")
print(true_classes[:20].reshape(1,20))

Epoch: 0 Loss: 0.5706942628553292 Accuracy: 0.8390892857142858
Epoch: 10 Loss: 0.26089711397556453 Accuracy: 0.9264464285714286
Epoch: 20 Loss: 0.2225320515159132 Accuracy: 0.9366607142857143
Epoch: 30 Loss: 0.19630437412406906 Accuracy: 0.9435178571428572
Epoch: 40 Loss: 0.18298849489942312 Accuracy: 0.9481964285714286
Epoch: 50 Loss: 0.17537373493681863 Accuracy: 0.9478928571428571
Epoch: 60 Loss: 0.16489533898476277 Accuracy: 0.9515178571428572
Epoch: 70 Loss: 0.16049785032910083 Accuracy: 0.9523392857142857
Epoch: 80 Loss: 0.15446537042788436 Accuracy: 0.9546428571428571
Epoch: 90 Loss: 0.15091692202882304 Accuracy: 0.9546964285714286

Final Accuracy: 0.9435

First 20 predictions:
[3 5 6 6 7 7 7 3 7 3 4 1 2 5 8 5 0 7 1 1]

First 20 actual:
[[8 5 6 6 7 7 7 2 7 3 4 1 2 5 8 3 0 7 1 1]]


In [40]:
weights, biases = model.init_parameters()

print("X:", X.shape)

for i, W in enumerate(weights):
    print(f"W{i+1}:", W.shape)

for i, b in enumerate(biases):
    print(f"b{i+1}:", b.shape)

X: (70000, 784)
W1: (784, 10)
W2: (10, 16)
W3: (16, 10)
b1: (10, 1)
b2: (16, 1)
b3: (10, 1)
